In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit import RDLogger
from mordred import Calculator, descriptors
from skfp.fingerprints import (
    MordredFingerprint, PubChemFingerprint, RDKitFingerprint,
    ECFPFingerprint, MACCSFingerprint, RDKit2DDescriptorsFingerprint,
    MAPFingerprint
)
from biosynfoni import Biosynfoni
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.warning')
pd.set_option("display.max_colwidth", 50)
tqdm.pandas()

In [ ]:
def mol_from_smiles(smiles_list):
    """Convert a list of SMILES strings to RDKit Mol objects with error handling."""
    mols = []
    for smi in tqdm(smiles_list, desc="Converting SMILES to Mol"):
        try:
            mols.append(Chem.MolFromSmiles(smi))
        except:
            mols.append(None)
    return mols

def compute_biosynfoni(mol):
    """Compute the Biosynfoni fingerprint for a single molecule."""
    try:
        return Biosynfoni(mol).fingerprint
    except:
        return None

def calculate_fingerprints(mols, fingerprint_config):
    """
    Compute multiple fingerprint types for a list of molecules.

    Args:
        mols: list of RDKit Mol objects
        fingerprint_config: dict mapping fingerprint names to their class constructors

    Returns:
        dict: fingerprint name as key and corresponding DataFrame as value
    """
    all_data = {}
    for name, fp_class in fingerprint_config.items():
        fp = fp_class()
        try:
            fp_desc = fp.transform(mols)
            feature_names = fp.get_feature_names_out()
            df = pd.DataFrame(fp_desc, columns=feature_names)
            all_data[name] = df
        except Exception as e:
            print(f"Error computing {name}: {e}")
    return all_data

def batch_process_fingerprints(smiles_list, fingerprint_config, batch_size=500, n_jobs=-1):
    """
    Process fingerprints in parallel batches for large datasets.

    Returns:
        dict: fingerprint name as key and concatenated DataFrame as value
    """
    batches = [smiles_list[i:i+batch_size] for i in range(0, len(smiles_list), batch_size)]
    
    def batch_fingerprint(batch_smiles):
        mols = [Chem.MolFromSmiles(smi) for smi in batch_smiles]
        return calculate_fingerprints(mols, fingerprint_config)
    
    batch_results = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(batch_fingerprint)(batch) for batch in batches
    )
    
    # Combine batch results
    final_results = {}
    for fp_type in batch_results[0].keys():
        dfs = [batch[fp_type] for batch in batch_results]
        final_results[fp_type] = pd.concat(dfs, ignore_index=True)
    
    return final_results

def compute_atom_bond_descriptors(mols, agg='sum'):
    """
    Compute atom and bond-level descriptors (available if needed).

    Args:
        mols: list of RDKit Mol objects
        agg: aggregation method ('sum', 'mean', or 'max')

    Returns:
        DataFrame with aggregated atom and bond features
    """
    ATOM_SYMBOLS = sorted({atom.GetSymbol() for mol in mols for atom in mol.GetAtoms()})
    HYBRIDIZATION_TYPE = [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3
    ]
    CHIRAL_TYPE = [
        Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW
    ]
    BOND_TYPES = [
        Chem.rdchem.BondType.SINGLE,
        Chem.rdchem.BondType.DOUBLE,
        Chem.rdchem.BondType.TRIPLE,
        Chem.rdchem.BondType.AROMATIC
    ]
    BOND_STEREO = [
        Chem.rdchem.BondStereo.STEREONONE,
        Chem.rdchem.BondStereo.STEREOE,
        Chem.rdchem.BondStereo.STEREOZ
    ]

    def one_hot_encode(value, allowable_values):
        return {f'{str(allowed)}': int(value == allowed) for allowed in allowable_values}

    def aggregate_features(features, agg_func):
        aggregated = {}
        for key in features[0]:
            aggregated[key] = agg_func([f[key] for f in features])
        return aggregated

    features_list = []
    for mol in mols:
        atom_features, bond_features = [], []
        for atom in mol.GetAtoms():
            atom_dict = {
                **one_hot_encode(atom.GetSymbol(), ATOM_SYMBOLS),
                **one_hot_encode(atom.GetHybridization(), HYBRIDIZATION_TYPE),
                **one_hot_encode(atom.GetChiralTag(), CHIRAL_TYPE),
                'degree': atom.GetDegree(),
                'total_hydrogens': atom.GetTotalNumHs(),
                'implicit_valence': atom.GetImplicitValence(),
                'aromatic': atom.GetIsAromatic(),
                'formal_charge': atom.GetFormalCharge(),
                'atomic_number': atom.GetAtomicNum(),
                'explicit_valence': atom.GetExplicitValence(),
                'num_radical_electrons': atom.GetNumRadicalElectrons()
            }
            atom_features.append(atom_dict)
        if mol.GetNumBonds() > 0:
            for bond in mol.GetBonds():
                bond_dict = {
                    **one_hot_encode(bond.GetBondType(), BOND_TYPES),
                    **one_hot_encode(bond.GetStereo(), BOND_STEREO),
                    'conjugated': bond.GetIsConjugated(),
                    'in_ring': bond.IsInRing()
                }
                bond_features.append(bond_dict)
        else:
            zero_features = {f'{bt}': 0 for bt in BOND_TYPES}
            zero_features.update({f'{bs}': 0 for bs in BOND_STEREO})
            zero_features['conjugated'] = 0
            zero_features['in_ring'] = 0
            bond_features.append(zero_features)

        agg_func = {'mean': np.mean, 'sum': np.sum, 'max': np.max}[agg]
        aggregated_atom_features = aggregate_features(atom_features, agg_func)
        aggregated_bond_features = aggregate_features(bond_features, agg_func)

        features_list.append(
            {**aggregated_atom_features, **aggregated_bond_features}
        )

    df = pd.DataFrame(features_list)
    df.columns = [f"atombond_{col}" for col in df.columns]
    return df


In [ ]:
print("Loading data...")
df1 = pd.read_csv("part_1.csv")
df2 = pd.read_csv("part_2.csv")
df3 = pd.read_csv("part_3.csv")
df = pd.concat([df1, df2, df3], axis=0).reset_index(drop=True)
print(f"ّFirst dataset shape: {df.shape}")

df4 = pd.read_csv("part_4.csv")
df5 = pd.read_csv("part_5.csv")
df6 = pd.read_csv("part_6.csv")
df_sec = pd.concat([df4, df5, df6], axis=0).reset_index(drop=True)
print(f"Secondary dataset shape: {df_sec.shape}")

In [ ]:
print("\nCalculating fingerprints for main dataset...")
mols = mol_from_smiles(df['smiles'].tolist())

In [ ]:
fingerprint_config = {
    'mordered': MordredFingerprint,
    'PubChem': PubChemFingerprint,
    'ECFP': lambda: ECFPFingerprint(radius=2),
    'rdkit-finger': RDKitFingerprint,
    'rdkit-dec': RDKit2DDescriptorsFingerprint,
    'Macss': MACCSFingerprint,
    'map': lambda: MAPFingerprint(fp_size=2048)
}

fingerprints = calculate_fingerprints(mols, fingerprint_config)

In [ ]:
for name, df_fp in fingerprints.items():
    df_fp.to_csv(f"{name}.csv", index=False)
    print(f"Saved {name}.csv with shape {df_fp.shape}")

In [ ]:
print("\nCalculating fingerprints for secondary dataset...")
mols_sec = mol_from_smiles(df_sec['smiles'].tolist())
fingerprints_sec = calculate_fingerprints(mols_sec, fingerprint_config)

# Save secondary fingerprints
for name, df_fp in fingerprints_sec.items():
    df_fp.to_csv(f"{name}2.csv", index=False)
    print(f"Saved {name}2.csv with shape {df_fp.shape}")

In [ ]:
print("\nCalculating Biosynfoni for secondary dataset...")
df_sec['mol'] = mols_sec
df_sec['Biosynfoni'] = df_sec['mol'].progress_apply(compute_biosynfoni)
df_biosynfoni = pd.DataFrame(df_sec['Biosynfoni'].to_list())
df_biosynfoni.columns = [f'Biosynfoni_{i+1}' for i in range(df_biosynfoni.shape[1])]
df_biosynfoni.to_csv("df_biosynfoni2.csv", index=False)
print("Saved Biosynfoni features.")

In [ ]:
print("\nBuilding final combined dataset...")
final_df = pd.concat([
    df_sec[['name', 'smiles', 'dataset']],
    fingerprints_sec['mordered'],
    fingerprints_sec['PubChem'],
    fingerprints_sec['ECFP'],
    fingerprints_sec['rdkit-finger'],
    fingerprints_sec['Macss'],
    fingerprints_sec['rdkit-dec'],
    df_biosynfoni,
    fingerprints_sec['map']
], axis=1)

print(f"Final dataset shape: {final_df.shape}")
final_df.to_csv("df_herbal2.csv", index=False)
print("Final dataset saved as 'df_herbal2.csv'")